# AI-Based Predictive Maintenance for Mining Equipment
**Group 9 · Saskatchewan Polytechnic · PROJ 613**

**Upload one file → recognize the input and its source format → resolve the equipment mapping → load its saved fault model → display the condition and probability.**

In VS Code, select the project’s `.venv` kernel and run all cells. If needed, install the upload widget in that kernel with `%pip install "ipywidgets>=8,<9"`, then restart the kernel. Use a complete, supported sensor recording or a supported fault-data image. Leave the mapping on **Automatic** for ordinary use. If you know the source of a shared-format file, choose its verified laboratory mapping before predicting.

Sensor CSVs support Ottawa, UODS, MCC5-THU, JUST and named CORA vibration recordings. See the README for the exact headers and rates. The original Ottawa and UODS sensor formats are also supported as MAT files. With **Automatic** selected, CORA inputs return **Ambiguous Equipment**. An event-camera image resolves automatically only when its exact bytes occur in one of the two trusted local export indexes and both indexes are available. An unmatched image returns **Unknown Equipment**. A Drill Rig render is recognized only if it matches the exact bytes of an indexed render under `Images/drill_rig/`. Dimensions alone cannot establish equipment identity.

Equipment names here describe **laboratory component mappings**. **Whole-machine photo** mode uses an optional local detector for excavators, wheel loaders and dump trucks. It reports visual identity only. A photo of a mining machine does not establish the condition of its internal parts. The Drill Rig images are CAD renders. Grader’s saved image evaluation concerns gear-tooth crops and has no upload checkpoint.

In [ ]:
from pathlib import Path
import csv
from functools import lru_cache
from hashlib import sha256
import re
import sys
from tempfile import TemporaryDirectory

import ipywidgets as widgets
from IPython.display import display

cwd = Path.cwd()
notebook_dir = next(
    (base / "Notebook" for base in (cwd, *cwd.parents)
     if (base / "Notebook" / "pdm.py").is_file()),
    None,
)
if notebook_dir is None:
    raise FileNotFoundError("Open this notebook from the capstone repository.")
sys.path.insert(0, str(notebook_dir))
import pdm

def checkpoint(machine, modality):
    suffix = {"Sensor": "sensor.joblib", "Image": "images.pt"}[modality]
    return pdm.MODELS / f"{machine}_{suffix}"

def checkpoint_status(machine, modality):
    if machine == "grader" and modality == "Image":
        return "Separate evaluation only"
    return "Found" if checkpoint(machine, modality).is_file() else "Missing"

availability = [
    {
        "Equipment": "Motor Grader" if machine == "grader" else config.title,
        "Sensor checkpoint": checkpoint_status(machine, "Sensor"),
        "Image checkpoint": checkpoint_status(machine, "Image"),
    }
    for machine, config in pdm.MACHINES.items()
]
display(pdm.pd.DataFrame(availability).style.hide(axis="index"))

### Upload and predict

Use trusted checkpoints from `Processed_Data/models/`. The notebook uses the saved sensor feature list and estimator, or the saved ResNet18 weights and the matching preprocessing in `pdm.py`. The repository’s final sensor training can select several algorithms, so the output reports the estimator actually saved. Confidence is the model’s class probability and has not been calibrated for field use. A checkpoint listed as found is validated when loaded.

The optional deployment commands are `python Notebook/pdm.py <equipment_key> train-final` for sensors and `python Notebook/pdm.py <equipment_key> train-final-images` for supported image mappings with exported training frames. They create local checkpoints, separate from the published held-out evaluations. The notebook never runs these commands. Shared formats remain ambiguous even if all candidate checkpoints are installed.

For a shared format, the optional mapping is a user-supplied provenance claim. The notebook shows this in the result and only accepts a candidate compatible with the detected format. CORA gear-wear runs are reused for Belt Conveyor and Dozer, and CORA drive runs overlap across Excavator and Wheel Loader. The sample alone cannot establish a mining machine's identity.

The [dataset manifest](../Dataset_Manifest/manifest.md) lists the training data for each mapping. Deployment fits can include the recordings used in the examples, so the sample predictions only verify that inference runs. Refer to the existing held-out results for performance estimates.

Local deployment builds for Drill Rig and Motor Grader use the first two seconds of each publisher sensor recording. The Drill Rig image build uses a small subset of publisher CAD meshes. These checkpoints have a different training scope from the archived evaluations. The fault output describes current component condition. It provides no estimate of future failure or remaining life.

Choose **Whole-machine photo** for ordinary site photos. The detector reports a category and detection score when it finds one supported category. It asks for a clearer photo when more than one supported category is visible or the score is too low. Its dump-truck label does not establish that the vehicle is a mining haul truck.

In [ ]:
@lru_cache(maxsize=3)
def indexed_image_hashes(folder):
    rows = pdm.read_index(folder)
    return {sha256((folder / row["file"]).read_bytes()).digest() for row in rows}


def identify_input(path, modality):
    if path.suffix.lower() == ".csv":
        with path.open(encoding="utf-8-sig", errors="replace") as stream:
            header = next(csv.reader([stream.readline(8192)]), [])
        schemas = [
            ("Ottawa bearing CSV", "pump", [pdm.OTTAWA_CSV_CHANNEL]),
            ("UODS bearing CSV", "haul_truck", pdm.UODS_CSV_CHANNELS),
            ("MCC5-THU gearbox", "drill_rig", pdm.MCC5_CHANNELS),
            ("JUST slewing bearing", "grader", pdm.JUST_CHANNELS),
        ]
        matches = [(source, machine) for source, machine, channels in schemas
                   if set(channels).issubset(header)]
        if matches:
            return " / ".join(source for source, _ in matches), [m for _, m in matches], {"mixed_schema": len(matches) > 1}
        speed = re.fullmatch(r"[A-Za-z0-9]+_(F5|F15|F50|F60)_S_Vx\.csv", path.name)
        if speed and len(header) == 1:
            try:
                value = float(header[0])
            except ValueError:
                pass
            else:
                if pdm.np.isfinite(value):
                    return "CORA rig vibration (Vx)", pdm.CORA, {"speed": speed.group(1)}
        return "Unrecognized CSV schema", [], {}

    try:
        source, machines, details = pdm.identify(path)
    except ValueError as problem:
        return str(problem), [], {}

    if modality == "Image" and machines == ["pump", "haul_truck"]:
        folders = {key: pdm.IMAGES / f"{key}_denseframes" for key in machines}
        if all((folder / "index.csv").is_file() for folder in folders.values()):
            digest = sha256(path.read_bytes()).digest()
            try:
                matches = [key for key, folder in folders.items() if digest in indexed_image_hashes(folder)]
            except OSError:
                matches = []
            if len(matches) == 1:
                return source, matches, {"indexed_image": True}

    if modality == "Image" and machines == ["drill_rig"]:
        trusted = pdm.IMAGES / "drill_rig"
        try:
            known_render = sha256(path.read_bytes()).digest() in indexed_image_hashes(trusted)
        except OSError:
            known_render = False
        if not known_render:
            return "Image provenance could not be verified", [], {}
        return source, machines, {"indexed_image": True}
    return source, machines, details


def score_checkpoint(path, machine, modality, source, details):
    weights = checkpoint(machine, modality)
    if modality == "Sensor":
        import joblib

        saved = joblib.load(weights)
        metadata = saved.get("metadata", {})
        if metadata and (metadata["machine"] != machine or metadata["modality"] != "sensor"):
            raise ValueError("Sensor checkpoint belongs to a different equipment or modality.")
        estimator = saved["model"]
        features = pdm.sensor_features(path, source, details)[saved["features"]]
        if features.empty:
            raise ValueError("The recording is too short for one complete feature window.")
        values = features.to_numpy(dtype=float)
        if not pdm.np.isfinite(values).all():
            raise ValueError("The recording produced invalid sensor features. Check the signal values.")
        scores = estimator.predict_proba(values).mean(axis=0)
        model_name = saved.get("model_name", type(estimator).__name__)
        return dict(zip(estimator.classes_, scores)), model_name

    return pdm.image_probability(path, machine), "ResNet18 CNN"


def predict_file(path, verified_mapping=None):
    extension = path.suffix.lower()
    if extension in {".csv", ".mat"}:
        modality = "Sensor"
    elif extension in {".png", ".jpg", ".jpeg"}:
        modality = "Image"
    else:
        return {"Status": "Unsupported file", "Accepted": "CSV, MAT, PNG, JPG, JPEG"}

    source, candidates, details = identify_input(path, modality)
    result = {"Input": modality, "Source": source}
    if not candidates:
        return {**result, "Status": "Unknown Equipment"}
    if verified_mapping is not None and verified_mapping not in candidates:
        return {**result, "Status": "Mapping incompatible with detected format"}
    if len(candidates) > 1:
        names = ", ".join(pdm.MACHINES[machine].title for machine in candidates)
        if verified_mapping is None or details.get("mixed_schema"):
            return {**result, "Status": "Ambiguous Equipment", "Candidates": names}
        result["Identification"] = "User-confirmed laboratory mapping"
    else:
        result["Identification"] = (
            "Exact match in a trusted local image export" if details.get("indexed_image")
            else "Recognized dataset format"
        )

    machine = verified_mapping or candidates[0]
    result["Equipment"] = "Motor Grader" if machine == "grader" else pdm.MACHINES[machine].title
    if not checkpoint(machine, modality).is_file():
        return {**result, "Status": "Model unavailable", "Required checkpoint": checkpoint(machine, modality).name}
    probabilities, model_name = score_checkpoint(path, machine, modality, source, details)
    fault = max(probabilities, key=probabilities.get)
    return {**result, "Status": "Prediction", "Predicted fault class": fault,
            "Model confidence": f"{probabilities[fault]:.1%}", "Model used": model_name}


@lru_cache(maxsize=1)
def photo_detector():
    import json

    folder = pdm.MODELS / "whole_machine"
    config_file = folder / "config.json"
    if not config_file.is_file():
        raise FileNotFoundError("Install the optional local whole-machine detector described in README.md.")
    config = json.loads(config_file.read_text())
    import onnxruntime

    session = onnxruntime.InferenceSession(str(folder / config["onnx"]), providers=["CPUExecutionProvider"])
    return session, config


def identify_photo(path):
    import cv2
    import numpy as np

    image = cv2.imread(str(path))
    if image is None:
        return {"Status": "Unknown Equipment", "Reason": "Upload a valid image."}
    session, config = photo_detector()
    size = config["imgsz"]
    height, width = image.shape[:2]
    scale = min(size / width, size / height)
    new_width, new_height = max(1, round(width * scale)), max(1, round(height * scale))
    dx, dy = (size - new_width) // 2, (size - new_height) // 2
    canvas = np.full((size, size, 3), config["letterbox_pad"], dtype=np.uint8)
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    canvas[dy:dy + new_height, dx:dx + new_width] = cv2.resize(rgb, (new_width, new_height))
    tensor = np.ascontiguousarray(canvas.transpose(2, 0, 1)[None], dtype=np.float32) / 255
    scores = session.run(None, {session.get_inputs()[0].name: tensor})[0][0, 4:]
    winners = scores.argmax(axis=0)
    candidates = {}
    for label in ("excavator", "wheel loader", "dump truck"):
        label_index = config["classes"].index(label)
        matches = scores[label_index, winners == label_index]
        if matches.size and matches.max() >= 0.60:
            candidates[label.title()] = float(matches.max())

    if not candidates:
        return {"Status": "Unknown Equipment", "Reason": "No supported category cleared the 60% score threshold."}
    if len(candidates) > 1:
        return {"Status": "Ambiguous Equipment", "Candidates": ", ".join(candidates)}
    equipment, score = next(iter(candidates.items()))
    return {"Status": "Equipment detected", "Equipment": equipment,
            "Detection score": f"{score:.1%}", "Model used": config["model"],
            "Fault prediction": "Unavailable for whole-machine photos"}

In [ ]:
upload = widgets.FileUpload(
    accept=".csv,.mat,.png,.jpg,.jpeg", multiple=False,
    description="Upload file",
)
mode = widgets.Dropdown(
    options=["Component fault", "Whole-machine photo"],
    description="Task",
)
mapping = widgets.Dropdown(
    options=[("Automatic", None)] + [(row["Equipment"], key)
                                     for key, row in zip(pdm.MACHINES, availability)],
    description="Mapping",
)
predict_button = widgets.Button(description="Predict", button_style="primary")
output = widgets.Output()

def on_predict(_):
    output.outputs = ()
    if not upload.value:
        output.append_stdout("Upload one file first.")
        return
    uploaded = upload.value[0]
    name = Path(uploaded["name"]).name
    if name in {"", ".", ".."}:
        output.append_stdout("Invalid filename.")
        return
    try:
        with TemporaryDirectory() as directory:
            path = Path(directory) / name
            path.write_bytes(uploaded["content"])
            if mode.value == "Whole-machine photo":
                result = identify_photo(path)
            else:
                result = predict_file(path, verified_mapping=mapping.value)
        output.append_stdout("\n".join(f"{label}: {value}" for label, value in result.items()))
    except Exception as problem:
        output.append_stdout(f"Prediction unavailable: {problem}")

predict_button.on_click(on_predict)
display(widgets.VBox([mode, mapping, widgets.HBox([upload, predict_button]), output]))